# Zero-Shot Large-Model Baseline (Qwen2.5-7B)

Evaluated a **7B general instruction model with no fine-tuning** on the same TSP test set, prompt, and metrics used for the fine-tuned 0.5B and 1.5B models. This
quantifies the value of task-specific fine-tuning: does a much larger general
model beat a small fine-tuned one?

**Before running:**
1. Change runtime type to **T4 GPU**.
2. Upload `test.jsonl` and `serialize.py` into `/content/`.

## 1. Installing dependencies

In [ ]:
!pip install -q -U "transformers>=4.46" accelerate bitsandbytes "protobuf>=5.26,<6"
!pip uninstall -q -y torchao

## 2. Loading Qwen2.5-7B-Instruct in 4-bit — **no adapter, no training**

The 7B model is loaded in 4-bit quantization so it fits a 16 GB T4 for inference.
There is no LoRA adapter.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL = "Qwen/Qwen2.5-7B-Instruct"

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16)

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(
    MODEL, quantization_config=bnb, device_map="auto", torch_dtype=torch.float16)
model.eval()
print("7B base model loaded (no fine-tuning)")

## 3. Helpers - identical prompt and parser to the fine-tuned runs

Using the same `SYSTEM_PROMPT`, `make_prompt`, and `parse_tour` as the fine-tuned
models is what makes the comparison fair.

In [ ]:
import json, numpy as np
from serialize import parse_tour, is_feasible, SYSTEM_PROMPT, make_prompt

def tour_length(coords, tour):
    pts = coords[tour]; nxt = np.roll(pts, -1, axis=0)
    return float(np.sqrt(((pts - nxt) ** 2).sum(axis=1)).sum())

def build_messages(coords):
    return [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": make_prompt(np.array(coords))}]

@torch.no_grad()
def generate_for(coords):
    msgs = build_messages(coords)
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = tokenizer(prompt, return_tensors='pt').to(model.device)
    out = model.generate(**inp, max_new_tokens=256, do_sample=True,
                         temperature=0.7, top_p=0.9, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True)

test = [json.loads(l) for l in open('/content/test.jsonl')]
print(len(test), "test instances")

## 4. Zero-shot evaluation, split by instance size

In [ ]:
small, large = [], []
for i, ex in enumerate(test):
    coords = np.array(ex['coords']); n = len(coords)
    tour = parse_tour(generate_for(coords), n)
    ok = is_feasible(tour, n)
    gap = (tour_length(coords, tour) - ex['opt_len'])/ex['opt_len']*100 if ok else None
    (small if n <= 13 else large).append((ok, gap))
    if (i+1) % 50 == 0:
        print(f"  ...{i+1}/{len(test)}")

for label, group in [("10-13 cities", small), ("14-20 cities", large)]:
    fr = sum(ok for ok,_ in group)/len(group)*100
    gaps = [g for ok,g in group if ok]
    mg = np.mean(gaps) if gaps else float('nan')
    print(f"{label}: feasibility {fr:.1f}%  |  gap {mg:.2f}%  ({len(group)} instances)")

## Result (recorded)

| Slice | Feasibility | Mean gap |
|-------|-------------|----------|
| 10–13 cities | 44.2% | 65.34% |
| 14–20 cities | 43.1% | 109.62% |

**Interpretation.** The 7B general model, zero-shot, produces a valid tour less
than half the time regardless of instance size — it understands the task but does
not reliably satisfy the output format and the all-cities constraint without
fine-tuning. The fine-tuned **1.5B** model (≈100% feasibility) clearly outperforms
it despite being ~4.7× smaller, demonstrating that task-specific fine-tuning beats
raw scale on this specialized task. (The fine-tuned 0.5B beats the 7B on small
instances but not large ones, where the 0.5B fails entirely — so capacity still
matters alongside fine-tuning.)